# Aura Inference Benchmarks

**Hands-on inference lab**: Deploy a model, serve inference, measure TTFT, KV cache, batching, quantization, and prefix caching.

**Requirements**: Google Colab with T4 GPU (free tier works) or any NVIDIA GPU with 16GB+ VRAM.

---

## Setup

Install dependencies and clone the repo.

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install vLLM and dependencies
!pip install vllm openai aiohttp matplotlib tabulate numpy pynvml pyyaml tqdm -q

In [ ]:
# Clone the repo
!git clone https://github.com/sshodhan/AuraInferenceBenchmarks.git 2>/dev/null || echo "Already cloned"
import os
os.chdir("AuraInferenceBenchmarks")
!pwd

## Start vLLM Server

We start the server in the background so we can run benchmarks in this notebook.

In [ ]:
import subprocess, time

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
PORT = 8000

# Start vLLM in background
server_proc = subprocess.Popen(
    ["python", "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--port", str(PORT)],
    stdout=open("/tmp/vllm_stdout.log", "w"),
    stderr=open("/tmp/vllm_stderr.log", "w"),
)
print(f"Started vLLM server (PID: {server_proc.pid})")
print(f"Model: {MODEL}")
print(f"Loading model... (this takes 1-3 minutes on Colab T4)")

In [ ]:
# Wait for server to be ready
from benchmarks.utils import wait_for_server

ready = wait_for_server(f"http://localhost:{PORT}/v1", timeout=300)
if ready:
    print("Server is ready! Proceeding to labs.")
else:
    print("Server failed to start. Check logs:")
    !tail -20 /tmp/vllm_stderr.log

In [ ]:
# Quick GPU check after model loading
!nvidia-smi

---
## Lab 1: Deploy Your First Model

Send test prompts and measure TTFT + generation time.

In [ ]:
from benchmarks.lab1_deploy import run_lab1
run_lab1()

### Lab 1 Reflection

**What to observe:**
- How long did model loading take? (Check the startup logs)
- What's the TTFT difference between short and long prompts?
- How much GPU memory does the model use?

---
## Lab 2: Measure KV Cache Impact

Send prompts of increasing length and watch TTFT + memory scale.

In [ ]:
from benchmarks.lab2_kvcache import run_lab2
run_lab2()

In [ ]:
# View the plots
from IPython.display import Image, display
display(Image(filename="results/lab2_ttft_vs_prompt_length.png"))
display(Image(filename="results/lab2_memory_vs_prompt_length.png"))

### Lab 2 Reflection

- Does TTFT scale linearly or super-linearly with prompt length?
- At what prompt length would you run out of GPU memory?
- How does this connect to KV cache capacity math?

---
## Lab 3: Batching & Throughput

Load test at different concurrency levels to see the throughput-latency tradeoff.

In [ ]:
from benchmarks.lab3_batching import run_lab3
run_lab3()

In [ ]:
# View the plots
from IPython.display import Image, display
display(Image(filename="results/lab3_concurrency_vs_throughput.png"))
display(Image(filename="results/lab3_concurrency_vs_per_request_tps.png"))

### Lab 3 Reflection

- At what concurrency does throughput stop improving?
- How much does per-request latency degrade with more concurrent users?
- This is the EXACT tradeoff you'd manage as TPM.

---
## Lab 4: Compare Model Sizes

**Note**: This lab stops the current server and starts new ones for each model. It takes longer.

In [ ]:
# Stop the current server first
server_proc.terminate()
server_proc.wait()
print("Server stopped.")
time.sleep(5)

In [ ]:
from benchmarks.lab4_model_comparison import run_lab4
import sys
sys.argv = ["lab4"]  # reset argv for argparse
run_lab4()

In [ ]:
# View comparison plots
from IPython.display import Image, display
display(Image(filename="results/lab4_ttft_comparison.png"))
display(Image(filename="results/lab4_memory_comparison.png"))

### Lab 4 Reflection

- Larger model = more memory for weights = less room for KV cache.
- This maps to Anthropic's Haiku (fast, cheap) vs Sonnet vs Opus (capable) tradeoffs.

---
## Lab 5: Quantization Impact

Compare FP16 vs INT8 (AWQ) — memory savings, speed changes, quality differences.

In [ ]:
from benchmarks.lab5_quantization import run_lab5
sys.argv = ["lab5"]  # reset argv
run_lab5()

In [ ]:
# View comparison
from IPython.display import Image, display
try:
    display(Image(filename="results/lab5_quantization_comparison.png"))
except FileNotFoundError:
    print("Plot not generated (only available when both modes are tested)")

### Lab 5 Reflection

- How much memory did quantization save?
- Was there a noticeable quality difference?
- This is a real production decision the inference team makes.

---
## Lab 6: Prompt Caching Simulation

Test prefix caching with a shared long system prompt across multiple requests.

In [ ]:
from benchmarks.lab6_prefix_caching import run_lab6
sys.argv = ["lab6", "--manage-servers"]  # test both modes
run_lab6()

In [ ]:
# View caching results
from IPython.display import Image, display
try:
    display(Image(filename="results/lab6_prefix_caching_ttft.png"))
    display(Image(filename="results/lab6_prefix_caching_bars.png"))
except FileNotFoundError:
    try:
        display(Image(filename="results/lab6_ttft_trend.png"))
    except FileNotFoundError:
        print("No plots generated")

### Lab 6 Reflection

- How much did TTFT drop with prefix caching enabled?
- This is how Claude Code iterations work efficiently: same codebase context, different questions.
- Directly relevant to AuraApp's nightly pipeline: shared style rules across all 40-60K requests.

---
## Summary & Interview Talking Points

In [ ]:
import json, glob

print("=" * 60)
print("  ALL LAB RESULTS")
print("=" * 60)

for result_file in sorted(glob.glob("results/lab*_results.json")):
    with open(result_file) as f:
        data = json.load(f)
    print(f"\n--- {data.get('lab', result_file)} ---")
    print(json.dumps({k: v for k, v in data.items() if k != 'raw'}, indent=2, default=str)[:500])

print("\n" + "=" * 60)
print("  Fill in your interview talking points:")
print("=" * 60)
print("""
1. "I deployed a model with vLLM and measured TTFT scaling from 100 to 4000 tokens —
    I saw firsthand how KV cache pressure drives memory usage."

2. "I load-tested inference at varying concurrency levels and observed the throughput-latency
    tradeoff — throughput plateaued at [X] concurrent requests when GPU became the bottleneck."

3. "I compared FP16 vs INT8 quantization and saw [X]% memory reduction with minimal quality
    impact — this is the kind of tradeoff I'd help the team evaluate for production."

4. "I tested prefix caching and saw TTFT drop by [X]% for repeated contexts — this directly
    maps to serving patterns like Claude Code where users iterate on the same codebase."
""")

---
## Cleanup

In [ ]:
# Kill any remaining vLLM servers
!pkill -f "vllm.entrypoints" 2>/dev/null || echo "No server to stop"
print("Done! Check the results/ directory for all saved data and plots.")